In [1]:
%%writefile practice10_3.cu
// Профилирование гибридного приложения CPU + GPU
// 1. реализовать гибридный алгоритм обработки массива данных;
// 2. использовать асинхронную передачу данных (cudaMemcpyAsync) и CUDA streams;
// 3. выполнить профилирование приложения:
  // a. определить накладные расходы передачи данных;
  // b. выявить узкие места при взаимодействии CPU и GPU;
// 4. предложить и реализовать одну оптимизацию, уменьшающую накладные расходы.
#include <iostream>                  // Для вывода результатов
#include <cuda_runtime.h>            // CUDA API
#include <vector>                    // Для использования vector
#include <chrono>                     // Для замера времени на CPU

#define N (1 << 20)                  // Размер массива (1 млн элементов)
#define BLOCK_SIZE 256               // Размер блока потоков CUDA


// CUDA-ядро для умножения на 2
__global__ void gpu_multiply(float* d_data, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Глобальный индекс потока
    if (idx < size)
        d_data[idx] *= 2.0f;                        // Умножаем элемент на 2
}


// Функция гибридной обработки CPU + GPU
int main() {
    size_t size_bytes = N * sizeof(float);                 // Размер массива в байтах
    std::vector<float> h_data(N, 1.0f);                   // Массив на CPU, инициализирован единицами

    float* d_data;
    cudaMalloc(&d_data, size_bytes);                       // Выделяем память на GPU

    cudaStream_t stream;                                   // Создаем CUDA stream для асинхронной передачи данных
    cudaStreamCreate(&stream);

    int half = N / 2;                                      // Разделяем данные: половина на CPU, половина на GPU


    // CPU часть
    auto cpu_start = std::chrono::high_resolution_clock::now(); // Замер времени CPU
    for (int i = 0; i < half; i++) {
        h_data[i] *= 2.0f;                                 // Умножаем на 2 на CPU
    }
    auto cpu_end = std::chrono::high_resolution_clock::now();
    double cpu_time = std::chrono::duration<double, std::milli>(cpu_end - cpu_start).count();


    // GPU часть (асинхронно)
    cudaMemcpyAsync(d_data, h_data.data() + half, half * sizeof(float), cudaMemcpyHostToDevice, stream); // Копируем вторую половину на GPU

    int blocks = (half + BLOCK_SIZE - 1) / BLOCK_SIZE;
    gpu_multiply<<<blocks, BLOCK_SIZE, 0, stream>>>(d_data, half); // Вычисления на GPU

    cudaMemcpyAsync(h_data.data() + half, d_data, half * sizeof(float), cudaMemcpyDeviceToHost, stream); // Возвращаем результаты на CPU

    cudaStreamSynchronize(stream);                        // Ждем завершения всех операций в stream


    // Вывод времени
    std::cout << "Время CPU обработки половины данных: " << cpu_time << " ms" << std::endl;

    // Простейшая проверка результата
    bool ok = true;
    for (int i = 0; i < N; i++) {
        if (h_data[i] != 2.0f) { ok = false; break; }
    }
    std::cout << "Проверка результата: " << (ok ? "OK" : "Ошибка") << std::endl;

    cudaFree(d_data);                                     // Освобождение памяти GPU
    cudaStreamDestroy(stream);                             // Удаление stream

    return 0;
}


Writing practice10_3.cu


In [2]:
# Компиляция
!nvcc practice10_3.cu -o practice10_3 -arch=sm_75 -std=c++11            # -arch=sm_75  - архитектура GPU (Tesla T4 в Colab = sm_75)
                                                                    # -std=c++11 — стандарт C++
# Запуск
!./practice10_3

Время CPU обработки половины данных: 1.50799 ms
Проверка результата: OK
